# Convolution Models

This notebook covers the convolutional building blocks and backbone models in ml_suite.

## Class hierarchy

```
ConvBlock                   <- primitive, 1D/2D/3D
    \-- ConditionedConvBlock  <- adds FiLM conditioning

ConvNet                     <- stacks ConvBlocks into a multi-stage backbone
    \-- ConditionedConvNet  <- replaces ConvBlocks with ConditionedConvBlocks
```

`ConvBlock` and `ConditionedConvBlock` are also used directly inside the UNet module:
- `UNetStage` / `UNetUpStage` (in `UNet`) use `ConvBlock`
- `UNetStage` / `UNetUpStage` (in `ConditionedUNet`) use `ConditionedConvBlock`

All classes are **dimension-agnostic** via `conv_dim=1/2/3`:

| `conv_dim` | Input shape | Real data examples |
|---|---|---|
| 1 | `(B, C, L)` | audio waveform, EEG, time series, 1D sensor |
| 2 | `(B, C, H, W)` | RGB image, spectrogram, 2D heatmap |
| 3 | `(B, C, D, H, W)` | CT/MRI volume, voxel grid, video clip |

## Task selection guide

| Task | Recommended class |
|---|---|
| Single conv + norm + activation (custom stack) | `ConvBlock` |
| Single conv conditioned on a global signal | `ConditionedConvBlock` |
| Feature extraction / classification | `ConvNet` |
| Feature extraction conditioned on a global signal | `ConditionedConvNet` |
| Spatial-to-spatial with same input/output size (segmentation, denoising) | `UNet` / `ConditionedUNet` (see notebook 03) |
| Diffusion noise predictor on spatial data | `ConditionedUNet(time_conditioning=True)` (see notebook 03) |

In [1]:
import torch
from ml_suite.models.convolution import (
    ConvBlock,
    ConditionedConvBlock,
    ConvNet,
    ConditionedConvNet,
)

---
## 1. `ConvBlock`  -  the primitive

A single convolution layer with optional normalisation, activation, and residual.

Operation order: **Conv -> Norm -> Residual Add -> Activation**

**Integrates into:**
- `ConvNet`  -  every block in every stage, including the stem
- `ConditionedConvNet.stem`  -  the entry convolution is always a plain `ConvBlock` even in the conditioned variant
- `UNetStage` / `UNetUpStage` (as `block_cls=ConvBlock`)  -  the per-block primitive inside `UNet`

**Use standalone when:** building a custom architecture that needs one (or a few) specific conv+norm+act steps  -  e.g. a custom stem, a projection between feature maps, or an entry block before a transformer.

**Do not use if:** you need multi-scale feature fusion (use `ConvNet` or `UNet`) or FiLM conditioning (use `ConditionedConvBlock`).

### 1.1 1D convolution

Processes sequential data with shape `(B, C, L)`. The most common starting point for audio, ECG, and time-series tasks.

**Integrates into:** `ConvNet` and `ConditionedConvNet` as the per-stage primitive block (all stages, including the stem).

**Use standalone when:** you need a single 1D conv+norm+act step — e.g. a custom entry block before a transformer encoder, or a projection between two 1D feature maps with different channel widths.

**Use for:** raw audio waveforms, ECG/EEG signals, IMU time series, 1D sensor streams.

In [2]:
block_1d = ConvBlock(
    input_channels=1, output_channels=32, conv_dim=1,
    kernel_size=3, padding=1, norm_type="batch", activation="silu",
)

# (batch, channels, length)
# Real: audio 16 kHz 1 s -> (B, 1, 16000),  ECG 12-lead -> (B, 12, 5000)
# padding=1 with kernel_size=3 keeps length unchanged (same-conv)
x_1d = torch.randn(2, 1, 256)
print(block_1d)
print(f"1D: {x_1d.shape} -> {block_1d(x_1d).shape}")   # (B, 32, 256)

SiLU(BatchNorm1d(Conv1D(1->32, k=(3,), s=(1,))))
1D: torch.Size([2, 1, 256]) -> torch.Size([2, 32, 256])


### 1.2 2D convolution

Processes spatial data with shape `(B, C, H, W)`. The standard choice for images, spectrograms, and 2D heatmaps.

**Integrates into:** `ConvNet` and `ConditionedConvNet` as the per-stage block; `UNetStage` / `UNetUpStage` with `block_cls=ConvBlock` inside `UNet`.

**Use standalone when:** you need a single 2D conv+norm+act step — e.g. a custom stem, a lateral connection between feature maps, or a projection head before global pooling.

**Use for:** RGB images, mel spectrograms, 2D heatmaps, X-ray / pathology slides, satellite imagery.

In [3]:
block_2d = ConvBlock(
    input_channels=3, output_channels=64, conv_dim=2,
    kernel_size=3, padding=1, norm_type="batch",
)

# (batch, channels, height, width)
# Real: RGB -> (B, 3, 224, 224),  mel spectrogram -> (B, 1, 128, 256)
# default activation="relu"; spatial size preserved by padding=1
x_2d = torch.randn(2, 3, 32, 32)
print(f"2D: {x_2d.shape} -> {block_2d(x_2d).shape}")   # (B, 64, 32, 32)

2D: torch.Size([2, 3, 32, 32]) -> torch.Size([2, 64, 32, 32])


### 1.3 3D convolution

Processes volumetric data with shape `(B, C, D, H, W)`. Computationally expensive — use only when depth axis carries genuine spatial meaning.

**Integrates into:** `ConvNet` and `ConditionedConvNet` with `conv_dim=3` as the per-stage block.

**Use standalone when:** you need a single 3D conv+norm+act step — e.g. a patch embedding for a 3D vision transformer, or a temporal smoothing block in a video pipeline.

**Use for:** CT / MRI volumes, voxel grids, video clips (time × height × width), 3D point-cloud projections.

In [4]:
block_3d = ConvBlock(
    input_channels=1, output_channels=16, conv_dim=3,
    kernel_size=3, padding=1,
    norm_type="group", num_groups=16,   # num_groups must divide output_channels
)

# (batch, channels, depth, height, width)
# Real: CT scan patch -> (B, 1, 64, 64, 64),  video clip -> (B, 3, 16, 112, 112)
# group norm preferred over batch norm for small-batch volumetric data
x_3d = torch.randn(2, 1, 16, 16, 16)
print(f"3D: {x_3d.shape} -> {block_3d(x_3d).shape}")   # (B, 16, 16, 16, 16)

3D: torch.Size([2, 1, 16, 16, 16]) -> torch.Size([2, 16, 16, 16, 16])


### 1.4 Normalisation types

Controls how activations are normalised after the convolution. The right choice depends on batch size and whether the model is used inside a diffusion pipeline.

**Integrates into:** every `ConvBlock` and `ConditionedConvBlock` used in `ConvNet`, `ConditionedConvNet`, and `UNet` / `ConditionedUNet`.

**Use standalone when:** configuring a custom `ConvBlock` stack — pick the norm type once and it applies to every block in that stack.

**Use for:** any conv-based model; the table below guides which to pick.

| `norm_type` | Normalises over | Best for |
|---|---|---|
| `"batch"` | batch x spatial | large batches, supervised classification |
| `"group"` | groups of channels per sample | small batches, diffusion models, UNet |
| `"layer"` | all channels per sample | batch-size-1, inference-only models |
| `None` | — | ablation, very shallow custom stacks |

**Inside `UNet` / `ConditionedUNet`:** `"group"` norm is standard — diffusion models typically use batch size 1-4 per GPU, where batch norm is unstable.

In [5]:
# (batch, channels, height, width) — same input for all variants
x = torch.randn(2, 64, 16, 16)

for norm in ["batch", "group", "layer", None]:
    kw = {"num_groups": 32} if norm == "group" else {}
    # num_groups=32 requires output_channels divisible by 32
    block = ConvBlock(input_channels=64, output_channels=64, conv_dim=2, norm_type=norm, **kw)
    out = block(x)
    # output shape is always (B, C_out, H, W) regardless of norm choice
    print(f"norm_type={str(norm):<8} -> {out.shape}")

norm_type=batch    -> torch.Size([2, 64, 16, 16])
norm_type=group    -> torch.Size([2, 64, 16, 16])
norm_type=layer    -> torch.Size([2, 64, 16, 16])
norm_type=None     -> torch.Size([2, 64, 16, 16])


### 1.5 Residual connection

Adds the input directly to the output: `out = Conv(x) + x`.  
Requires `input_channels == output_channels` and compatible spatial size (stride=1).

**Integrates into:** `ConvNet` — the first block of each stage never uses a residual (channel count changes); all subsequent same-width blocks within a stage get `do_residual=True` automatically.

**Use standalone when:** manually constructing a stack of same-width `ConvBlock`s and you want residual shortcuts — e.g. a custom ResNet-style body or a refinement module appended to a pretrained backbone.

**Use for:** any setting where gradient flow through many blocks matters — deep stacks (≥4 blocks), fine-tuning, or diffusion model refinement steps.

In [6]:
block = ConvBlock(
    input_channels=64, output_channels=64,   # must match for residual add
    conv_dim=2, do_residual=True,
    norm_type="group", num_groups=32,
)

# (batch, channels, height, width)
# Real: feature map mid-backbone -> (B, 64, 56, 56) for ImageNet stem output
# stride must be 1 (default) — stride > 1 changes spatial size, breaking the residual add
x   = torch.randn(2, 64, 16, 16)
out = block(x)
print(f"Residual: {x.shape} -> {out.shape}")   # (B, 64, 16, 16) — shape preserved

Residual: torch.Size([2, 64, 16, 16]) -> torch.Size([2, 64, 16, 16])


---
## 2. `ConditionedConvBlock`  -  FiLM-conditioned building block

Extends `ConvBlock` with FiLM: a global context vector `(B, context_dim)` produces per-channel scale and shift.

**Integrates into:**
- `ConditionedConvNet`  -  every stage block (not the stem) is a `ConditionedConvBlock`
- `UNetStage` / `UNetUpStage` with `block_cls=ConditionedConvBlock`  -  how `ConditionedUNet` conditions its conv layers

**Use standalone when:** you need a single conditioned conv step  -  e.g., a FiLM adapter inserted into a pretrained network. The FiLM projection is **zero-initialised**, so the block starts as a plain `ConvBlock` and conditioning strength is learned from scratch.

**How a user is meant to use it:** you rarely construct `ConditionedConvBlock` directly. Instead, use `ConditionedConvNet` or `ConditionedUNet` which wire this class into a complete model. Build it manually only when assembling a custom hybrid architecture.

In [7]:
# 2D conditioned block
cblock = ConditionedConvBlock(
    input_channels=64, output_channels=64,
    context_dim=128,   # e.g. diffusion time embedding
    conv_dim=2, norm_type="group", num_groups=32, do_residual=True,
)

x   = torch.randn(2, 64, 16, 16)
ctx = torch.randn(2, 128)   # global condition  -  (batch, context_dim)
# Real: sinusoidal time embedding (B, 256), class embedding (B, 128), CLIP pooled (B, 768)

out = cblock(x, ctx)
print(f"2D FiLM: x={x.shape}, ctx={ctx.shape} -> {out.shape}")

# 1D conditioned block
cblock_1d = ConditionedConvBlock(
    input_channels=32, output_channels=32,
    context_dim=64, conv_dim=1, norm_type="group", num_groups=32,
)
x_1d = torch.randn(2, 32, 128)
ctx_1d = torch.randn(2, 64)
print(f"1D FiLM: x={x_1d.shape}, ctx={ctx_1d.shape} -> {cblock_1d(x_1d, ctx_1d).shape}")

2D FiLM: x=torch.Size([2, 64, 16, 16]), ctx=torch.Size([2, 128]) -> torch.Size([2, 64, 16, 16])
1D FiLM: x=torch.Size([2, 32, 128]), ctx=torch.Size([2, 64]) -> torch.Size([2, 32, 128])


---
## 3. `ConvNet`  -  multi-stage backbone

A complete 1D/2D/3D feature extractor or classifier built from stacked `ConvBlock` stages with progressive downsampling.

**Integrates into:** subclassed by `ConditionedConvNet`, which overrides `_make_conv_block` to swap in `ConditionedConvBlock`.

**How a user is meant to use it:**
- `num_classes=None` -> use as an **encoder** (e.g. before a transformer, an MLP head, or a decoder)
- `num_classes=N` -> use as a **standalone classifier** (adds global pool + linear head)
- Do NOT use `ConvNet` when you need the output to be the same spatial size as the input  -  use `UNet` for that.

**Returned tensor:**
- Feature extractor mode (`num_classes=None`): `(B, C_last, *spatial)`  -  the final feature map before pooling
- Classifier mode (`num_classes=N`): `(B, N)`  -  class logits

### 3.1 1D feature extractor

Stacks `ConvBlock`s over a 1D sequence, progressively expanding channels and halving the length at each stage boundary.

**Integrates into:** can be used as-is as the encoder trunk before an MLP head, a transformer, or a decoder. Subclassed by `ConditionedConvNet` (section 4).

**Use standalone when:** you need a pure feature extractor or classifier for sequential data and no spatial reconstruction is needed downstream.

**Use for:** ECG / EEG backbone, raw waveform encoder for ASR or speech synthesis, time-series representation learning, pre-encoder before a sequence transformer.

**Applications:** ECG arrhythmia classification, human activity recognition from IMU, keyword spotting, anomaly detection on sensor streams.

In [8]:
net_1d = ConvNet(
    conv_dim=1, in_channels=1,
    stage_channels=[32, 64, 128],
    blocks_per_stage=2,
    norm_type="batch",
    num_classes=None,   # feature extractor — feed output to a downstream head
)

# (batch, channels, length)
# Real: ECG 12-lead -> (B, 12, 5000);  raw audio mono -> (B, 1, 16000)
x   = torch.randn(2, 1, 256)
out = net_1d(x)
# 3 stages, 2 downsamples (stem has no stride) -> length / 4;  final channels = 128
print(f"1D feature map: {x.shape} -> {out.shape}")   # (B, 128, 64)

1D feature map: torch.Size([2, 1, 256]) -> torch.Size([2, 128, 64])


### 3.2 2D image classifier

Stacks `ConvBlock`s over spatial feature maps, progressively expanding channels and halving H×W at each stage boundary, then collapses with global pooling before a linear head.

**Integrates into:** can be used as a drop-in image backbone before any downstream head. Subclassed by `ConditionedConvNet` (section 4).

**Use standalone when:** you need a classifier or feature extractor for 2D spatial data and spatial resolution does not need to be preserved in the output.

**Use for:** image classification, X-ray / pathology diagnosis, defect detection, satellite image classification, spectrogram-based audio classification.

**Applications:** CIFAR / ImageNet classification, chest X-ray screening, PCB defect detection, land-use classification from satellite.

In [9]:
net_2d = ConvNet(
    conv_dim=2, in_channels=3,
    stage_channels=[64, 128, 256, 512],
    blocks_per_stage=2,
    norm_type="batch",
    global_pool_mode="avg",
    num_classes=10,
    activation="silu",
)

# (batch, channels, height, width)
# Real: CIFAR-10 -> (B, 3, 32, 32);  ImageNet -> (B, 3, 224, 224)
# 4 stages -> 3 stride-2 downsamples (stem stride=1) -> H,W / 8 before global pool
x      = torch.randn(2, 3, 32, 32)
logits = net_2d(x)
print(f"2D classifier: {x.shape} -> {logits.shape}")   # (B, 10)

2D classifier: torch.Size([2, 3, 32, 32]) -> torch.Size([2, 10])


### 3.3 3D volumetric backbone

Stacks `ConvBlock`s over volumetric data, halving D×H×W at each stage boundary. Memory and compute grow as O(D×H×W) — keep patch sizes modest or use `blocks_per_stage=1` for deep nets.

**Integrates into:** can be used as the encoder trunk before a 3D segmentation decoder or a classification head. Subclassed by `ConditionedConvNet` with `conv_dim=3`.

**Use standalone when:** you need a 3D classifier or volumetric feature extractor and full spatial reconstruction is not needed.

**Use for:** 3D medical image classification, action recognition from video, voxel-based shape analysis.

**Applications:** brain tumour / lesion classification from MRI, lung nodule detection from CT, action recognition from video clips.

In [10]:
net_3d = ConvNet(
    conv_dim=3, in_channels=1,
    stage_channels=[16, 32, 64],
    blocks_per_stage=2,
    norm_type="group", num_groups=16,   # group norm stable at batch_size=1
    global_pool_mode="avg",
    num_classes=4,
)

# (batch, channels, depth, height, width)
# Real: brain MRI patch -> (B, 1, 64, 64, 64);  lung CT ROI -> (B, 1, 32, 128, 128)
# small batch (B=1) is typical for 3D — batch norm would be unstable here
x      = torch.randn(1, 1, 16, 16, 16)
logits = net_3d(x)
print(f"3D classifier: {x.shape} -> {logits.shape}")   # (B, 4)

3D classifier: torch.Size([1, 1, 16, 16, 16]) -> torch.Size([1, 4])


### 3.4 Downsample modes: `stride`, `maxpool`, `avgpool`

Controls how spatial resolution is halved at each stage boundary. The mechanism affects gradient flow and learned representations but not the output shape.

**Integrates into:** every `ConvNet` and `ConditionedConvNet` — set once at construction, applies uniformly to all stage boundaries.

**Use standalone when:** benchmarking architecture variants or choosing a mode for a specific deployment (e.g. `"stride"` for learned downsampling, `"maxpool"` for classic VGG / ResNet behaviour).

**Use for:** any `ConvNet` usage — see the table for guidance on when each mode is appropriate.

| `downsample_mode` | Mechanism | Trade-off |
|---|---|---|
| `"stride"` (default) | stride-2 conv on first block of each stage | Learnable downsampling; slightly more params; modern default |
| `"maxpool"` | stride-1 conv + MaxPool(2x2) | Fixed max pooling; classic ResNet/VGG style |
| `"avgpool"` | stride-1 conv + AvgPool(2x2) | Fixed average pooling; smoother gradients |

In [11]:
# (batch, channels, height, width)
# Real: RGB -> (B, 3, 32, 32);  spectrogram -> (B, 1, 128, 256)
x = torch.randn(2, 3, 32, 32)

for mode in ["stride", "maxpool", "avgpool"]:
    net = ConvNet(
        conv_dim=2, in_channels=3,
        stage_channels=[32, 64, 128], blocks_per_stage=1,
        downsample_mode=mode, num_classes=5,
    )
    n = sum(p.numel() for p in net.parameters())
    # all three modes produce identical output shape — differ only in downsampling op
    print(f"downsample_mode='{mode}': {net(x).shape}, params={n:,}")

downsample_mode='stride': torch.Size([2, 5]), params=103,653
downsample_mode='maxpool': torch.Size([2, 5]), params=103,653
downsample_mode='avgpool': torch.Size([2, 5]), params=103,653


### 3.5 Global pooling modes

Controls how the final feature map is collapsed before the classification head. Only used when `num_classes` is set; in feature-extractor mode the feature map is returned directly.

**Integrates into:** the global pooling stage of `ConvNet` and `ConditionedConvNet` when `num_classes` is not `None`.

**Use standalone when:** choosing a pooling strategy for a classifier — `"cat_avg_max"` gives a richer representation at the cost of a wider linear head.

**Use for:** any `ConvNet` / `ConditionedConvNet` classifier configuration.

| `global_pool_mode` | Output channels | Notes |
|---|---|---|
| `"avg"` (default) | `C_last` | Standard — good general-purpose choice |
| `"max"` | `C_last` | Better for sparse / peaked features |
| `"cat_avg_max"` | `2 x C_last` | Concatenates both — richer, slightly more params |

> Only used when `num_classes` is set. In feature-extractor mode the feature map is returned directly.

In [12]:
# (batch, channels, height, width)
# Real: RGB image -> (B, 3, 32, 32)
x = torch.randn(2, 3, 32, 32)

for mode in ["avg", "max", "cat_avg_max"]:
    net = ConvNet(
        conv_dim=2, in_channels=3,
        stage_channels=[32, 64], blocks_per_stage=1,
        global_pool_mode=mode, num_classes=10,
    )
    # cat_avg_max concatenates avg and max pools -> 2×C_last features -> wider linear head
    print(f"global_pool_mode='{mode}': {net(x).shape}")   # always (B, 10)

global_pool_mode='avg': torch.Size([2, 10])
global_pool_mode='max': torch.Size([2, 10])
global_pool_mode='cat_avg_max': torch.Size([2, 10])


### 3.6 Per-stage block counts

Passing a list to `blocks_per_stage` gives independent depth control per stage. Mirrors many efficient architectures that use shallow early stages (large spatial size — expensive) and deeper later stages (small spatial size — cheap).

**Integrates into:** `ConvNet` and `ConditionedConvNet` — any integer list of length `len(stage_channels)` is valid.

**Use standalone when:** tailoring the depth profile of a `ConvNet` for a specific compute budget or task — e.g. allocating more blocks to later stages for fine-grained recognition.

**Use for:** any `ConvNet` / `ConditionedConvNet` where uniform depth per stage is sub-optimal, such as large-input classifiers or efficient mobile architectures.

In [13]:
net = ConvNet(
    conv_dim=2, in_channels=3,
    stage_channels=[32, 64, 128, 256],
    blocks_per_stage=[1, 2, 3, 3],   # more depth at higher-level features
    num_classes=100,
    norm_type="layer",
    global_pool_mode="cat_avg_max"
)

# (batch, channels, height, width)
# Real: CIFAR-100 -> (B, 3, 32, 32);  fine-grained recognition -> (B, 3, 224, 224)
# layer norm chosen here; switch to "group" for diffusion or small-batch settings
x   = torch.randn(2, 3, 32, 32)
out = net(x)
print(f"Per-stage depths [1,2,3,3]: {out.shape}")   # (B, 100)
print(net)

Per-stage depths [1,2,3,3]: torch.Size([2, 100])
ConvNet 2D (in_channels=3)
  ├── Stem: SiLU(GroupNorm(Conv2D(3->32, k=(3, 3), s=(1, 1))))
  ├── Stage 1 (1 blocks):
  │     └── SiLU(x + GroupNorm(Conv2D(32->32, k=(3, 3), s=(1, 1))))
  ├── Stage 2 (2 blocks):
  │     └── SiLU(GroupNorm(Conv2D(32->64, k=(3, 3), s=(2, 2))))
  │     └── SiLU(x + GroupNorm(Conv2D(64->64, k=(3, 3), s=(1, 1))))
  ├── Stage 3 (3 blocks):
  │     └── SiLU(GroupNorm(Conv2D(64->128, k=(3, 3), s=(2, 2))))
  │     └── SiLU(x + GroupNorm(Conv2D(128->128, k=(3, 3), s=(1, 1))))
  │     └── SiLU(x + GroupNorm(Conv2D(128->128, k=(3, 3), s=(1, 1))))
  ├── Stage 4 (3 blocks):
  │     └── SiLU(GroupNorm(Conv2D(128->256, k=(3, 3), s=(2, 2))))
  │     └── SiLU(x + GroupNorm(Conv2D(256->256, k=(3, 3), s=(1, 1))))
  │     └── SiLU(x + GroupNorm(Conv2D(256->256, k=(3, 3), s=(1, 1))))
  ├── Global Pooling: CAT_AVG_MAX
  └── Head: Linear(512->100)


### 3.7 Receptive field diagnostic

`ConvNet.print_receptive_field()` computes the analytical receptive field (RF) and cumulative stride after each layer. Use this to verify that the network can "see" enough spatial context for a given task before committing to a training run.

**Integrates into:** available on any `ConvNet` or `ConditionedConvNet` instance — diagnostic only, does not affect the forward pass.

**Use standalone when:** designing the stage layout for a new task and you need to confirm the RF covers the objects or patterns of interest.

**Use for:** any `ConvNet` architecture design workflow — compare the final cumulative RF to your input resolution and the scale of features to detect.

**How to pick:** if the final RF is smaller than the target object size, add more stages, increase `kernel_size`, or use dilated convolutions.

In [14]:
net = ConvNet(
    conv_dim=2, in_channels=3,
    stage_channels=[32, 64, 128, 256],
    blocks_per_stage=[1, 2, 3, 3],   # more depth at higher-level features
    num_classes=100,
    norm_type="layer",
    global_pool_mode="cat_avg_max"
)
# RF grows with every block; stride doubles at each stage boundary
# final cumulative stride = 8 for 4 stages (2^3 from stage 2, 3, 4 entries)
# last-stage RF = 71 pixels — adequate for 32×32 inputs; scale blocks for larger inputs
net.print_receptive_field()

Layer / Module                                | Layer RF   | Cumulative RF   | Cumulative Stride
-----------------------------------------------------------------------------------------------
Stem                                          | 3          | 3.0             | 1.0            
Stage 1 - Block 0                             | 3          | 5.0             | 1.0            
Stage 2 - Block 0                             | 3          | 7.0             | 2.0            
Stage 2 - Block 1                             | 3          | 11.0            | 2.0            
Stage 3 - Block 0                             | 3          | 15.0            | 4.0            
Stage 3 - Block 1                             | 3          | 23.0            | 4.0            
Stage 3 - Block 2                             | 3          | 31.0            | 4.0            
Stage 4 - Block 0                             | 3          | 39.0            | 8.0            
Stage 4 - Block 1                             |

### 3.8 Conv type: `standard` vs `separable`

Separable convolutions factorize the spatial and channel operations, dramatically reducing FLOPs and parameters at the cost of a minor quality trade-off.

**Integrates into:** every block in `ConvNet` and `ConditionedConvNet` including the stem; `ConditionedConvNet` with `conv_type="separable"` uses `SeparableConditionedConvBlock` in all stages.

**Use standalone when:** deploying to mobile or edge hardware, or training wide networks (large `stage_channels`) on a FLOPs budget.

**Use for:** mobile-class classifiers, on-device feature extractors, efficient conditioned backbones.

| `conv_type` | Mechanism | When to use |
|---|---|---|
| `"standard"` (default) | Full `k x k` convolution | Best quality; default choice |
| `"separable"` | Depthwise (spatial) + Pointwise 1x1 (channel) | Mobile/edge inference; FLOPs budget; large kernels |

**Savings rule:** ~`k^2` fewer multiply-adds per block (9x at `kernel_size=3`). Worthwhile when output channels are wide (>=64). At narrow widths (e.g. C=8) the overhead of the pointwise negates most of the gain.

In [15]:
# (batch, channels, height, width)
x = torch.randn(2, 3, 32, 32)
params = []
for conv_type in ["standard", "separable"]:
    net = ConvNet(
        conv_dim=2, in_channels=3,
        stage_channels=[64, 128, 256], blocks_per_stage=2,
        num_classes=10, conv_type=conv_type,
    )
    n = sum(p.numel() for p in net.parameters())
    params.append(n)
    # separable = depthwise k×k (spatial) + pointwise 1×1 (channel mix)
    print(f"conv_type='{conv_type}': {net(x).shape}, params={n:,}")   # (B, 10)

# Parameter savings, and FLOP savings
parameter_savings = (params[0] - params[1]) / params[0]
print(f"Parameter savings: {parameter_savings:.2%}")
# Flop ratio ~ 1 /N + 1/Dk^2, N = channels, Dk = kernel size for each layer
flop_ratio = sum(
    (1 / c + 1 / (k ** 2)) for c, k in zip([64, 128, 256], [3, 3, 3])
)
print(f"FLOP ratio: {flop_ratio:.2%}")

conv_type='standard': torch.Size([2, 10]), params=1,186,826
conv_type='separable': torch.Size([2, 10]), params=143,784
Parameter savings: 87.88%
FLOP ratio: 36.07%


---
## 4. `ConditionedConvNet`  -  FiLM-conditioned backbone

Subclasses `ConvNet` and overrides `_make_conv_block` to use `ConditionedConvBlock` in all stages.  
The stem remains a plain `ConvBlock` (context is not applied to the entry conv).

**Integrates into:** Nothing

**How a user is meant to use it:** use this when you need a `ConvNet`-style backbone but want the feature extraction to adapt to a global conditioning signal. The interface is identical to `ConvNet` except `forward` also takes `context`.

**When to use `ConditionedConvNet` vs `ConditionedUNet`:**
- `ConditionedConvNet` is a **classifier / feature extractor**  -  it progressively downsamples and discards spatial resolution
- `ConditionedUNet` is a **spatial-to-spatial** model  -  it preserves input spatial size through skip connections

**Applications:** class-conditioned feature extraction, sensor-conditioned prediction, cross-modal backbone (audio features conditioning a video backbone).

### 4.1 2D conditioned feature extractor

Demonstrates the most common use of `ConditionedConvNet`: extracting spatial features from an image while adapting to a global context vector via FiLM conditioning.

**Integrates into:** use directly as an encoder before a classification head or a decoder; combine with a diffusion time embedding to get a conditioned feature extractor for latent diffusion.

**Use standalone when:** you need a `ConvNet`-style 2D backbone that adapts its feature responses to a global signal — e.g. class label, style vector, or cross-modal embedding.

**Use for:** class-conditioned feature extraction, style-conditioned backbone, cross-modal image encoding (e.g. audio-conditioned image features).

In [16]:
net = ConditionedConvNet(
    conv_dim=2, in_channels=3,
    stage_channels=[64, 128, 256],
    blocks_per_stage=2,
    context_dim=128,
    norm_type="group", num_groups=32,
    num_classes=None,   # encoder mode — returns feature map, not logits
)

# (batch, channels, height, width)
# Real: RGB image -> (B, 3, 224, 224);  satellite tile -> (B, 4, 64, 64)
x   = torch.randn(2, 3, 32, 32)
ctx = torch.randn(2, 128)   # (batch, context_dim)
# Real: class label embedding (B, 128), audio embedding (B, 512), style vector (B, 64)
features = net(x, ctx)
print(f"Conditioned features: x={x.shape}, ctx={ctx.shape} -> {features.shape}")   # (B, 256, 8, 8)

Conditioned features: x=torch.Size([2, 3, 32, 32]), ctx=torch.Size([2, 128]) -> torch.Size([2, 256, 8, 8])


### 4.2 1D conditioned classifier

Subject-conditioned human activity recognition (HAR): a per-subject embedding adapts the feature extractor to individual physiological differences, improving generalisation across subjects.

**Integrates into:** use as a drop-in conditioned 1D classifier; the context vector can be any global embedding broadcast over the sequence.

**Use standalone when:** you have sequential sensor data and a global per-sample attribute (subject ID, device type, recording protocol) that should modulate the feature extraction.

**Use for:** subject-adaptive HAR from IMU, speaker-conditioned speech feature extraction, patient-conditioned ECG analysis.

In [17]:
net = ConditionedConvNet(
    conv_dim=1, in_channels=6,   # 6 IMU axes (acc x/y/z + gyro x/y/z)
    stage_channels=[32, 64, 128],
    blocks_per_stage=2,
    context_dim=32,
    global_pool_mode="avg",
    num_classes=6,
)

# (batch, channels, length)
# Real: IMU recording -> (B, 6, 500)  (6 axes, 500 timesteps at 100 Hz = 5 s)
# per-subject embedding shifts feature responses via FiLM scale/shift
x      = torch.randn(2, 6, 128)
ctx    = torch.randn(2, 32)   # per-subject embedding — (batch, context_dim)
logits = net(x, ctx)
print(f"Conditioned 1D classifier: {logits.shape}")   # (B, 6)

Conditioned 1D classifier: torch.Size([2, 6])


### 4.3 3D conditioned backbone

Protocol-conditioned MRI analysis: acquisition parameters (field strength, TR/TE) form a context vector that shifts feature responses, helping the model adapt to scanner variation without retraining.

**Integrates into:** use as an encoder for 3D conditioned classification or as the backbone before a 3D segmentation decoder.

**Use standalone when:** you have volumetric data paired with a global acquisition or patient metadata vector that should modulate feature extraction.

**Use for:** scanner-protocol-conditioned MRI analysis, subject-conditioned 3D action recognition, condition-adaptive CT lesion detection.

In [18]:
net = ConditionedConvNet(
    conv_dim=3, in_channels=1,
    stage_channels=[16, 32, 64],
    blocks_per_stage=2,
    context_dim=16,
    norm_type="group", num_groups=16,   # group norm stable for small 3D batches
    global_pool_mode="cat_avg_max",     # concatenates avg + max -> 2×C_last features
    num_classes=3,
)

# (batch, channels, depth, height, width)
# Real: brain MRI patch -> (B, 1, 64, 64, 64);  scanner meta (field, TR, TE) -> (B, 16)
# FiLM shifts feature responses per-scanner without retraining
x      = torch.randn(1, 1, 16, 16, 16)
ctx    = torch.randn(1, 16)   # scanner acquisition metadata
logits = net(x, ctx)
print(f"Conditioned 3D classifier: {logits.shape}")   # (B, 3)

Conditioned 3D classifier: torch.Size([1, 3])


### 4.4 Separable conditioned backbone

Same interface as `ConditionedConvNet` — only `conv_type` changes. Use when FLOPs are a constraint (mobile / edge deployment) but you still need FiLM conditioning.

**Integrates into:** drop-in replacement for `ConditionedConvNet` wherever parameter and compute budgets are tight.

**Use standalone when:** deploying a conditioned backbone to edge hardware, or when training large batches with wide channels and FLOPs are the bottleneck.

**Use for:** mobile-class conditioned image classifiers, on-device conditioned ECG analysis, efficient style-conditioned feature extraction.

In [19]:
# (batch, channels, height, width)
# Real: RGB image -> (B, 3, 224, 224);  style vector -> (B, 64)
x   = torch.randn(2, 3, 32, 32)
ctx = torch.randn(2, 64)   # (batch, context_dim)

for conv_type in ["standard", "separable"]:
    net = ConditionedConvNet(
        conv_dim=2, in_channels=3,
        stage_channels=[64, 128, 256], blocks_per_stage=2,
        context_dim=64, num_classes=10, conv_type=conv_type,
    )
    n = sum(p.numel() for p in net.parameters())
    # separable saves ~87% params while keeping the same FiLM conditioning interface
    print(f"conv_type='{conv_type}': {net(x, ctx).shape}, params={n:,}")   # (B, 10)

conv_type='standard': torch.Size([2, 10]), params=1,303,306
conv_type='separable': torch.Size([2, 10]), params=260,264
